<a href="https://colab.research.google.com/github/a-nushkasharma/LLM_Analysis_Wrapper/blob/main/VulnerabilityAnalyzer_v5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM-based Smart Contract Analyzer

## 1. Dependencies


In [ ]:
# packages
!pip install -q -U bitsandbytes
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U git+https://github.com/huggingface/accelerate.git
!pip install -q -U git+https://github.com/huggingface/peft.git
!pip install -q -U datasets


# libraries
!pip install -q -U sentence-transformers
!pip install -q streamlit streamlit-mermaid
!pip install -q google-generativeai openai
!pip install -q langchain langchain-community faiss-cpu
!pip install -q -U langchain-text-splitters langchain-huggingface langchain-core
!pip install -q googlesearch-python beautifulsoup4 requests
!pip install googlesearch-python beautifulsoup4 langchain_core
!pip install -q pyngrok python-dotenv

## 2. Set API keys
1.  huggingface (Your Hugging Face Read Token for Gemma)
2.  GEMINI_API_KEY
3.  OPEN_API_KEY
4.  NGROK_AUTH_TOKEN (for connecting backend with frontend)



# 3. Project Directories

In [ ]:
!mkdir -p prompt_templates
!mkdir -p uploads
!mkdir -p outputs
!mkdir -p .streamlit

### Front-end theme


In [ ]:
%%writefile .streamlit/config.toml
[theme]
# Primary accent color for interactive elements (buttons, etc).
primaryColor="#0068C9"

# Background color for the main content area.
backgroundColor="#FFFFFF"

# Background color for sidebar and most interactive widgets.
secondaryBackgroundColor="#F0F2F6"

# Color used for almost all text.
textColor="#0C0D0F"

# Font family for all text in the app.
font="sans serif"

## 4. Prompt Files

In [ ]:
%%writefile prompt_templates/gemma_prompt.txt
<start_of_turn>user
Below is an instruction that describes a task. Write a response that appropriately completes the request.
List all the vulnerabilities in the following solidity code of smart contract: <end_of_turn> <start_of_turn>model

In [ ]:
%%writefile prompt_templates/phase1.txt
You are a smart contract security auditor. You will be given the FULL contract code.
You will also be provided with a **preliminary analysis** generated by another AI model (Gemma) retrieved from our knowledge base.
Your task is to:
1.  Perform your own full, independent analysis of the contract.
2.  **Review the Preliminary Analysis:** Check if the vulnerabilities listed in the context are valid.
3.  **Validate or Reject:** If Gemma found a bug that is real, include it. If it's a hallucination, ignore it.
4.  **Find New Issues:** Find any vulnerabilities that Gemma missed.

**Preliminary Analysis Context (from RAG):**
{gemma_analysis_report}

**Your Response Rules:**
1.  Your **entire** output must be a single, valid JSON array.
2.  Do **not** include any text, explanations, or markdown fences (like ```json) before or after the JSON array.
3.  If you find multiple vulnerabilities, ensure each JSON object in the array is separated by a comma.
4.  **Strictly use the official SWC ID** (e.g., 'SWC-107' for Reentrancy, 'SWC-114' for Front-running) for the `'category'` field whenever applicable.

**Example of a valid response with multiple findings:**
`[{{"id": "vuln-001", "category": "SWC-107", ...}}, {{"id": "vuln-002", "category": "SWC-101", ...}}]`

For each vulnerability, return a JSON object with the following structure:
```json
{{
  "id": "vuln-<short-id>",
  "title": "<short name>",
  "category": "<The official SWC ID or descriptive name>",
  "severity": "low|medium|high|critical",
  "confidence": 0.0-1.0,
  "evidence": "<exact code line(s) or snippet>",
  "rationale": "<why this is an issue>",
  "affected_components": ["function names / storage vars"],
  "recommendation": "<fix or mitigation>",
  "related_refs": []
}}
If no vulnerabilities are found, return an empty JSON array : []

In [ ]:
%%writefile prompt_templates/phase2.txt
You are a smart contract security auditor performing a cross-verification step.
You will be given the FULL contract code and a `counterparty_report` from another LLM.

Your task is to use the provided `counterparty_report` to **refine, verify, or correct** the findings.

**Your response must follow these rules strictly:**
1.  Your **ENTIRE** output must be a single, valid JSON array, containing your reviewed findings.
2.  Do **NOT** add any commentary, explanations, or markdown (like ```json) before or after the JSON.
3.  If no vulnerabilities from the counterparty report are confirmed, return an empty JSON array: `[]`.
4.  **Strictly use the official SWC ID** (e.g., 'SWC-107', 'SWC-114') for the `'category'` field. Correct the category if the counterparty report used the wrong one.

---
### **Inputs You Will Be Given:**

1.  **Counterparty report (JSON):**
    {counterparty_report}

---
### **Required Output Format:**
```json
{{
  "id": "<same id as counterparty finding>",
  "title": "<short name, possibly refined>",
  "category": "<The official SWC ID or descriptive name>",
  "severity": "low|medium|high|critical",
  "confidence": 0.0-1.0,
  "evidence": "<exact code line(s) or snippet>",
  "rationale": "<why this is an issue>",
  "affected_components": ["function names / storage vars"],
  "recommendation": "<fix or mitigation>",
  "related_refs": []
}}

In [ ]:
%%writefile prompt_templates/phase3.txt
You are a lead security analyst performing a final consensus check. Your task is to review the verification report (`other_findings`) from the previous phase (Phase 2) and either **CONFIRM** or **DISPUTE** its conclusion regarding each vulnerability.

**Your response must follow these rules strictly:**
1.  Your **ENTIRE** output must be a single, valid JSON array, containing one object for each finding reviewed.
2.  Do **NOT** add any commentary, explanations, or markdown fences (like ```json) before or after the JSON array. Your response must start with `[` and end with `]`.
3.  Base your decision solely on the data within the provided verification findings (`other_findings`).
4.  If the previous phase correctly identified the vulnerability *and* used the correct SWC ID, set `stance` to `confirm`.
5.  If the previous phase identified a vulnerability but used an **incorrect SWC ID**, set `stance` to `confirm` but provide the correct ID in `corrected_category`.
6.  If the previous phase's finding seems incorrect or lacks sufficient evidence, set `stance` to `dispute`.

---
### **Input You Will Be Given:**

1.  **Prior Verification Findings (from Phase 2):**
    {other_findings}

---
### **Required Output Format:**
```json
{{
  "id": "<same id from the findings>",
  "stance": "confirm|dispute",
  "confidence": 0.0-1.0,
  "reason": "a concise justification for your stance, potentially mentioning category correction",
  "corrected_category": "<The correct SWC ID if Phase 2 was wrong, otherwise null>",
  "what_is_missing": "<if dispute, specify what evidence is needed; else null>"
}}

In [ ]:
%%writefile prompt_templates/phase4.txt
You are a master security analyst performing a final, high-stakes verification. You will be given a piece of code that is suspected to be vulnerable, along with a patched "counter-example" that is known to be safe.

Your task is to perform a differential analysis to determine with high certainty if the original vulnerability exists.

**Your response must follow these rules strictly:**
1.  Your **ENTIRE** output must be a single, valid JSON object.
2.  If 'vulnerability_confirmed', you **MUST** provide a `control_flow_graph` and trace the `vulnerable_flow_steps`.
3.  The `control_flow_graph` **MUST** use simple `graph LR;` (Left-to-Right) flowchart syntax.
4.  Do **NOT** include any natural language explanations, comments, or markdown fences *inside* the `control_flow_graph` string.
5.  When creating a loop or link, link back to the node's **ID only** (e.g., `C --> B;`). Do **NOT** re-declare the node's label (e.g., DO NOT do `C --> B[Verify Balance];`).

---
### **Inputs for Analysis:**

1.  **Original Vulnerable Code Snippet:**
    {vulnerable_code}

2.  **Initial Rationale for Vulnerability:**
    {initial_rationale}

3.  **Patched Counter-Example (Known Safe Version):**
    {patched_code}

4.  **Explanation of Fix:**
    {fix_explanation}

---
### **Required Output Format:**
```json
{{
  "id": "{id}",
  "final_decision": "vulnerability_confirmed|vulnerability_rejected",
  "detailed_reasoning": "A step-by-step explanation of why the original code is vulnerable by comparing it directly to the safe counter-example. Explain the difference in execution flow.",
  "control_flow_graph": "<A simple, valid Mermaid syntax string. Example: graph LR; A[Start] --> B{{Decision?}}; B -- Yes --> C[Action]; C --> B; B -- No --> D[End];>",
  "vulnerable_flow_steps": [
    // Include one or more step objects as needed to trace the full exploit path.
    {{
      "step": 1,
      "description": "<Description of the first step in the exploit>",
      "code_block": "<The exact code block for this step>"
    }},
    {{
      "step": 2,
      "description": "<Description of the second step>",
      "code_block": "<The exact code block for this step>"
    }}
    // ... add more steps if necessary
  ]
}}


In [ ]:
%%writefile prompt_templates/phase5.txt
You are an expert code analyst. Extract a secure Solidity code snippet and a brief explanation for fixing the specified vulnerability from the provided text.
Respond ONLY with a valid JSON object like this:
{{ "code_snippet": "...", "explanation": "..." }}

Vulnerability to fix: {vuln_name}

Text to analyze:
{page_text}

## 5. Counter-Examples library

In [ ]:
%%writefile counter_examples.json
{
  "SWC-104": {
    "name": "Unchecked Low-Level Call",
    "fix_pattern": "Check the boolean return value",
    "explanation": "Low-level functions like `.call()`, `.delegatecall()`, and `.send()` return a boolean `success` flag. Secure code must check this flag and revert the transaction with a `require(success, ...)` statement if the call fails. This prevents the contract from reaching an inconsistent state where an external interaction fails silently.",
    "code_snippet": "contract SecureCall {\n    function executeCall(address payable _to, uint256 _amount) public {\n        // The return value of the low-level .call() is checked.\n        (bool success, ) = _to.call{value: _amount}(\"\");\n        require(success, \"External call failed\");\n    }\n}"
  },
  "SWC-107": {
    "name": "Re-entrancy",
    "fix_pattern": "Checks-Effects-Interactions (CEI) Pattern",
    "explanation": "The patched version performs the state change (updating the balance) *before* the external call. This prevents a malicious contract from re-entering the function and exploiting the contract's state before it's updated.",
    "code_snippet": "contract SecureBank {\n    mapping(address => uint) public balances;\n\n    function withdraw() public {\n        // 1. Checks: Verify the condition first.\n        uint amount = balances[msg.sender];\n        require(amount > 0, \"No balance to withdraw.\");\n\n        // 2. Effects: Update the state immediately.\n        balances[msg.sender] = 0;\n\n        // 3. Interactions: Send the funds last.\n        (bool success,) = msg.sender.call{value: amount}(\"\");\n        require(success, \"Transfer failed.\");\n    }\n}"
  },
  "SWC-106": {
    "name": "Suicidal Contracts",
    "fix_pattern": "onlyOwner Modifier",
    "explanation": "The `selfdestruct()` function, which removes a contract from the blockchain, is protected by an access control modifier like `onlyOwner`. This ensures that only the authorized owner of the contract can call this critical function, preventing malicious destruction.",
    "code_snippet": "contract SecureDestruction is Ownable {\n    // The 'onlyOwner' modifier ensures only the owner can call this.\n    function destroy() public onlyOwner {\n        // The contract's funds are safely sent to the owner.\n        selfdestruct(payable(owner()));\n    }\n}"
  },
    "VULN-GREEDY": {
    "name": "Greedy Contracts",
    "fix_pattern": "Add a secure withdrawal function",
    "explanation": "Greedy contracts can accept Ether but have no function to allow for its withdrawal, effectively locking the funds forever. The fix is to add an explicit withdrawal function, protected by a modifier like `onlyOwner`, so an authorized party can retrieve the contract's balance.",
    "code_snippet": "contract SecurePiggyBank is Ownable {\n    receive() external payable {}\n\n    // An explicit withdrawal function is added.\n    // The 'onlyOwner' modifier ensures only the owner can call it.\n    function withdraw() public onlyOwner {\n        // Transfer the entire balance of the contract to the owner.\n        payable(owner()).transfer(address(this).balance);\n    }\n}"
  },
  "VULN-PRODIGAL": {
    "name": "Prodigal Contracts",
    "fix_pattern": "State Management",
    "explanation": "The contract uses a state variable, typically a mapping, to track if a user has already performed an action (like claiming a reward). By checking and updating this state variable, the contract prevents unauthorized users from draining funds by repeating the action.",
    "code_snippet": "contract SecureAirdrop {\n    uint public rewardAmount = 0.1 ether;\n    // This mapping tracks who has already claimed.\n    mapping(address => bool) public hasClaimed;\n\n    constructor() payable {}\n\n    function claimReward() public {\n        require(address(this).balance >= rewardAmount, \"Not enough funds.\");\n        // Check if the user has already claimed.\n        require(!hasClaimed[msg.sender], \"You have already claimed your reward.\");\n\n        // Mark the user as claimed BEFORE sending funds.\n        hasClaimed[msg.sender] = true;\n        payable(msg.sender).transfer(rewardAmount);\n    }\n}"
  },
  "SWC-101": {
    "name": "Integer Overflow and Underflow",
    "fix_pattern": "Use Solidity >=0.8.0",
    "explanation": "Modern Solidity compilers (version 0.8.0 and newer) have built-in checks that will cause a transaction to revert if an integer overflow or underflow occurs, making arithmetic operations safe by default.",
    "code_snippet": "contract SafeMathContract {\n    // In Solidity 0.8+, this function will revert on overflow automatically.\n    function add(uint a, uint b) public pure returns (uint) {\n        return a + b;\n    }\n}"
  },
  "VULN-TYPECAST": {
    "name": "Address Typecasting",
    "fix_pattern": "_safeMint for ERC721",
    "explanation": "When minting NFTs, using `_safeMint` instead of `_mint` is crucial. `_safeMint` includes a check to verify if the recipient address is a smart contract and, if so, whether it can handle ERC721 tokens. This prevents NFTs from being permanently locked in contracts that cannot manage them.",
    "code_snippet": "contract SecureMinter is ERC721 {\n    uint256 private _nextTokenId;\n\n    constructor() ERC721(\"Secure Token\", \"SAFE\") {}\n\n    function awardItem(address player) public returns (uint256) {\n        uint256 newItemId = _nextTokenId++;\n        // _safeMint checks if a contract recipient can handle ERC721 tokens.\n        _safeMint(player, newItemId);\n        return newItemId;\n    }\n}"
  },
  "VULN-MAP-WRITE": {
    "name": "Mapping Read/Write Vulnerabilities",
    "fix_pattern": "Restricted write-access",
    "explanation": "This happens when a contract allows unauthorized users to modify a public mapping, which can lead to takeovers or broken logic. The fix is to restrict write access to these critical functions using modifiers like `onlyOwner` or requiring that only existing members can add new ones.",
    "code_snippet": "contract SecureAdmin is Ownable {\n    mapping(address => bool) public isAdmin;\n\n    constructor() {\n        // The deployer is the first admin and the owner.\n        isAdmin[owner()] = true;\n    }\n\n    // Only the contract owner can add new admins.\n    function addAdmin(address _newAdmin) public onlyOwner {\n        isAdmin[_newAdmin] = true;\n    }\n\n    function executeCriticalTask() public {\n        require(isAdmin[msg.sender], \"Not an admin!\");\n        // execute task\n    }\n}"
  },
  "SWC-114": {
    "name": "Transaction Order Dependence (Front-running)",
    "fix_pattern": "Commit-Reveal Scheme",
    "explanation": "The user's action is split into two phases. First, they commit to a cryptographic hash of their intended action plus a secret value ('salt'). After the commit is confirmed, they reveal the actual data and salt. The contract verifies the reveal against the commit. By the time an attacker sees the revealed data, it's too late to front-run the action.",
    "code_snippet": "contract SecureGuessingGame {\n    bytes32 private immutable SOLUTION_HASH = keccak256(abi.encodePacked(\"my_secret_word\"));\n    mapping(address => bytes32) public commitments;\n\n    // Step 1: User commits to a hashed answer with a secret \"salt\".\n    function commit(bytes32 _commitment) public {\n        require(commitments[msg.sender] == bytes32(0), \"You have already committed.\");\n        commitments[msg.sender] = _commitment;\n    }\n\n    // Step 2: User reveals their original answer and the salt.\n    function reveal(string calldata solution, bytes32 salt) public {\n        bytes32 regeneratedCommitment = keccak256(abi.encodePacked(solution, salt));\n        require(regeneratedCommitment == commitments[msg.sender], \"Revealed data does not match commitment.\");\n        // ... game logic ...\n    }\n}"
  },
  "VULN-TX-STATE": {
    "name": "Transaction State Dependence",
    "fix_pattern": "Slippage Protection",
    "explanation": "This occurs when a transaction's outcome depends on a state variable that a privileged user can change mid-action. The fix is to implement slippage protection, where the user specifies the minimum acceptable outcome (e.g., minAmountOut), and the contract reverts if the final result is worse than this expectation.",
    "code_snippet": "contract SecureWithdrawal is Ownable {\n    mapping(address => uint) public balances;\n    uint public withdrawalFee = 1;\n\n    // The user must specify the minimum amount they expect to receive.\n    function withdraw(uint amount, uint minAmountOut) public {\n        require(balances[msg.sender] >= amount, \"Insufficient balance.\");\n        uint feeAmount = (amount * withdrawalFee) / 100;\n        uint finalAmount = amount - feeAmount;\n\n        // The contract verifies the outcome meets the user's expectation.\n        require(finalAmount >= minAmountOut, \"High fee change, withdrawal reverted.\");\n        balances[msg.sender] -= amount;\n        payable(msg.sender).transfer(finalAmount);\n    }\n}"
  },
  "SWC-120": {
    "name": "Block State Dependence",
    "fix_pattern": "VRF Oracle",
    "explanation": "Instead of using predictable or manipulable block properties like `block.timestamp` for randomness, the contract requests a verifiably random number from a decentralized oracle service like Chainlink VRF. The oracle provides a secure random number in a callback function, which prevents miners from influencing the outcome.",
    "code_snippet": "contract SecureLottery is VRFConsumerBaseV2 {\n    // ... (Chainlink VRF setup) ...\n    address[] public s_players;\n    address public s_winner;\n    uint256 public s_requestId;\n\n    // Step 1: Request a random number from the oracle.\n    function pickWinner() public {\n        require(s_players.length > 0, \"No players.\");\n        s_requestId = VRF_COORDINATOR.requestRandomWords(/*...*/);\n    }\n\n    // Step 2: The oracle calls this function back with the secure random number.\n    function fulfillRandomWords(uint256 requestId, uint256[] memory randomWords) internal override {\n        uint256 randomIndex = randomWords[0] % s_players.length;\n        s_winner = s_players[randomIndex];\n    }\n}"
  },
  "VULN-MEM-OVERLAP": {
    "name": "Memory Overlap and Unsafe Access",
    "fix_pattern": "Avoid manual memory management",
    "explanation": "This occurs when using low-level inline assembly to manipulate memory directly. Mistakes in calculating memory pointers can cause one variable's data to overwrite another. The safest counter-example is to avoid manual memory management and rely on Solidity's built-in memory handling.",
    "code_snippet": "contract SafeMemoryUsage {\n    // By using standard Solidity types and functions,\n    // the compiler handles memory allocation safely,\n    // preventing overlap issues that can arise from\n    // manual pointer manipulation in inline assembly.\n    function processData(uint[] calldata data) external pure returns (uint) {\n        uint sum = 0;\n        for (uint i = 0; i < data.length; i++) {\n            sum += data[i];\n        }\n        return sum;\n    }\n}"
  },
    "VULN-NO-FALLBACK": {
    "name": "Payable Function Without Fallback",
    "fix_pattern": "Implement the receive() function",
    "explanation": "If a contract needs to accept plain Ether transfers (without a function call), it must implement the `receive() external payable {}` function. Without it, direct transfers of Ether to the contract address will fail, potentially locking funds.",
    "code_snippet": "contract SecureReceiver {\n    event Received(address sender, uint amount);\n\n    function deposit() public payable {\n        // Logic for when deposit is called directly\n    }\n\n    // This function explicitly handles plain Ether transfers.\n    receive() external payable {\n        emit Received(msg.sender, msg.value);\n    }\n}"
  },
  "SWC-116": {
    "name": "Time Manipulation",
    "fix_pattern": "Use oracles or block numbers for time",
    "explanation": "The `block.timestamp` can be manipulated by a miner to some extent, making it unreliable for critical logic that depends on precise timing. The secure approach is to use a decentralized oracle for a reliable timestamp or measure time in terms of block numbers, which are more difficult to manipulate.",
    "code_snippet": "contract SecureTimeLock {\n    uint public unlockBlockNumber;\n    uint constant LOCK_DURATION_IN_BLOCKS = 100; // e.g., approx 20 minutes\n\n    constructor() {\n        unlockBlockNumber = block.number + LOCK_DURATION_IN_BLOCKS;\n    }\n\n    function withdraw() public {\n        require(block.number >= unlockBlockNumber, \"Lock period not over.\");\n        // ... withdrawal logic ...\n    }\n}"
  },
    "VULN-ACCESS-CONTROL": {
    "name": "Improper Access Control",
    "fix_pattern": "onlyOwner Modifier / Role-Based Access",
    "explanation": "This vulnerability occurs when critical functions that should be restricted (like changing ownership or withdrawing treasury funds) are left `public` without any checks. The fix is to protect these functions with access control modifiers, such as `onlyOwner`, which verify that `msg.sender` is an authorized address before execution.",
    "code_snippet": "contract ControlledAccess is Ownable {\n    // A critical function that should be protected.\n    function changeCriticalSetting(uint256 _newSetting) public onlyOwner {\n        // Only the owner can change this setting.\n        // ... logic ...\n    }\n}"
  },
  "SWC-113": {
    "name": "Denial of Service (DoS)",
    "fix_pattern": "Pull-over-Push Pattern",
    "explanation": "A DoS can occur if a contract pushes payments to an array of users in a loop; one malicious user whose address reverts can block the entire function forever. The secure 'pull-over-push' pattern allows each user to call a `claim()` function to withdraw their funds individually. This isolates failures and ensures the contract remains operational for all other users.",
    "code_snippet": "contract SecureDistributor {\n    mapping(address => uint) public rewards;\n\n    // Instead of pushing funds, users pull them.\n    function claimReward() public {\n        uint amount = rewards[msg.sender];\n        require(amount > 0, \"No reward to claim.\");\n        rewards[msg.sender] = 0;\n        (bool success, ) = msg.sender.call{value: amount}(\"\");\n        require(success, \"Transfer failed.\");\n    }\n}"
  }
}

## 6. utils.py

In [ ]:
%%writefile utils.py
import json
import os
from typing import Any, Dict, List

def load_prompt(filename: str, prompt_dir: str = "prompt_templates", context: Dict[str, Any] = None) -> str:
    """
    Load a prompt template file and format it with the provided context.
    """
    path = os.path.join(prompt_dir, filename)
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Prompt file not found: {path}")

    with open(path, "r", encoding="utf-8") as f:
        prompt_template = f.read()


    if context:
        try:
            # Ensure lists of dicts etc are stringified as JSON
            formatted_context = {k: json.dumps(v, indent=2) if isinstance(v, (dict, list)) else v for k, v in context.items()}
            return prompt_template.format(**formatted_context)
        except KeyError as e:
            print(f"❌ KeyError: The placeholder {{{e.args[0]}}} was not found in the context dictionary for prompt '{filename}'.")
            return prompt_template
        except Exception as e:
            print(f"❌ Error formatting prompt {filename}: {e}")
            return prompt_template

    return prompt_template

def save_json(data: dict, filepath: str):
    """
    Save a Python dictionary as a JSON file.
    Creates directories if they do not exist.
    """
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

## 7. rag_store.py

In [ ]:
%%writefile rag_store.py
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
import os
from typing import List

class Retriever:
    def __init__(self, index_path="faiss_index"):
        self.index_path = index_path
        # Force embeddings to CPU to save GPU for Gemma
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2",
            model_kwargs={'device': 'cpu'}
        )
        self.registry = {}
        self.db = self._load_db()

    def _load_db(self):
        try:
            if os.path.exists(self.index_path):
                return FAISS.load_local(self.index_path, self.embeddings, allow_dangerous_deserialization=True)
                print("Loaded existing FAISS index from disk.")
        except Exception as e:
            print(f"Could not load existing FAISS index: {e}. A new one will be created.")
        return None

    def index_contract(self, contract_id: str, contract_data: str):
        if contract_id in self.registry:
            print(f"Contract ID '{contract_id}' already indexed.")
            return

        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=100,
            length_function=len
        )
        chunks = text_splitter.split_text(contract_data)

        documents = [Document(page_content=chunk, metadata={"source": contract_id}) for chunk in chunks]

        try:
            # Re-initialize DB if None or append if exists
            if self.db is None:
                self.db = FAISS.from_documents(documents, self.embeddings)
            else:
                self.db.add_documents(documents)

            self.db.save_local(self.index_path)
            self.registry[contract_id] = True
            print(f"Successfully indexed contract {contract_id} and saved to {self.index_path}.")
        except Exception as e:
            print(f"Error creating/saving FAISS index: {e}")


    def retrieve(self, contract_id: str, query: str, k: int = 5) -> List[str]:
        if self.db is None:
            print("FAISS index not loaded. Cannot retrieve.")
            return []

        try:
            # Search specifically within the contract if possible, or general search
            docs = self.db.similarity_search(query, k=k)
            return [doc.page_content for doc in docs]
        except Exception as e:
            print(f"Error during FAISS retrieval: {e}")
            return []

## 8. gemma_client.py

In [ ]:
%%writefile gemma_client.py
import os
import json
import torch
from utils import load_prompt
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "weifar/FTAudit-gemma-7b-v1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

class GemmaClient:
    def __init__(self, prompt_dir: str = "prompt_templates"):
        self.hf_token = userdata.get("huggingface")
        if not self.hf_token:
            raise ValueError("Missing Hugging Face Token (huggingface) in Colab Secrets")

        print("Loading 4-bit fine-tuned Gemma model... This may take a few minutes.")

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map={"":0},
            token=self.hf_token
        )

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            add_eos_token=True,
            padding_side='left',
            token=self.hf_token
        )
        self.prompt_dir = prompt_dir
        self.device = "cuda:0"
        print("Gemma model loaded successfully on T4 GPU.")

    def _get_prompt(self) -> str:
        filename = "gemma_prompt.txt"
        return load_prompt(filename, self.prompt_dir)

    def analyze(self, contract_code: str) -> str:
        """
        Runs the vulnerability analysis and returns the RAW TEXT response.
        """
        prompt_template = self._get_prompt()

        # the specific prompt format for the ft model
        prompt = f"{prompt_template}\n```\n{contract_code}\n```"

        encodeds = self.tokenizer(prompt, return_tensors="pt", add_special_tokens=True)
        model_inputs = encodeds.to(self.device)

        try:
            print("Gemma analysis running (Text Mode)...")
            generated_ids = self.model.generate(
                **model_inputs,
                max_new_tokens=4096,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )

            decoded = self.tokenizer.decode(generated_ids[0], skip_special_tokens=True)


            # model usually echoes the prompt so split by a specific token
            if "smart contract:" in decoded:
                response_text = decoded.split("smart contract:")[-1].strip()
            else:
                response_text = decoded

            # cleaning gemma response
            if response_text.startswith("model"):
                response_text = response_text[5:].strip()

            print(f"Gemma Analysis Length: {len(response_text)} chars")
            return response_text

        except Exception as e:
            print(f"⚠️ Error calling Gemma model: {e}")
            return "Gemma analysis failed."

## 9. llm1_api.py



In [ ]:
%%writefile llm1_api.py
import os
import json
import openai
from dotenv import load_dotenv
from typing import Any, Dict
from utils import load_prompt
from google.colab import userdata

load_dotenv()

class LLM1Client:
    def __init__(self, api_key: str = None, model: str = None, prompt_dir: str = "prompt_templates"):
        self.api_key = api_key or os.getenv("OPENAI_API_KEY") or userdata.get("OPENAI_API_KEY")
        if not self.api_key:
            raise ValueError("Missing OpenAI API Key")

        self.model = model or os.getenv("OPENAI_MODEL", "gpt-4o-mini")
        self.client = openai.OpenAI(api_key=self.api_key)
        self.prompt_dir = prompt_dir

    def _get_prompt(self, phase: int, context: Dict[str, Any] = None) -> str:
        filename = f"phase{phase}.txt"
        return load_prompt(filename, self.prompt_dir, context=context)

    def analyze_contract(self, contract_code: str, phase: int = 1, context: Dict[str, Any] = None) -> list | dict:

        if phase == 5 and context and "custom_prompt" in context:
            full_prompt = context["custom_prompt"]
        else:
            # Only load the file if we are not using a custom prompt
            try:
                prompt_with_context = self._get_prompt(phase, context)
                full_prompt = f"{prompt_with_context}\n\nContract Code:\n{contract_code}"
            except Exception as e:
                print(f"⚠️ Error loading prompt for Phase {phase}: {e}")
                return []

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": "You are a helpful security auditor. Your entire response MUST be a single valid JSON object."},
                    {"role": "user", "content": full_prompt}
                ]
            )
            raw_text = response.choices[0].message.content
            print(f"--- LLM1 Raw Output (Phase {phase}) ---")
            print(raw_text)

            parsed_json = json.loads(raw_text)
            print(f"--- LLM1 Parsed JSON (Phase {phase}) --- Type: {type(parsed_json)}")

            final_output = []

            if isinstance(parsed_json, dict):
                found_nested_list = False
                for key in ["vulnerabilities", "findings", "results"]:
                    if key in parsed_json and isinstance(parsed_json[key], list):
                        print(f"--- LLM1 Extracting list from key '{key}' (Phase {phase}) ---")
                        final_output = parsed_json[key]
                        found_nested_list = True
                        break

                if not found_nested_list:
                    print(f"--- LLM1 No nested list found. Assuming dict is the finding. Wrapping in list. (Phase {phase}) ---")
                    final_output = [parsed_json]

            elif isinstance(parsed_json, list):
                 print(f"--- LLM1 Received list as expected (Phase {phase}) ---")
                 final_output = parsed_json
            else:
                 print(f"--- LLM1 Unexpected type received: {type(parsed_json)}. Returning empty list. ---")
                 final_output = []

            print(f"--- LLM1 Final Return Value (Phase {phase}) --- Type: {type(final_output)}")
            return final_output

        except json.JSONDecodeError as json_err:
             print(f"⚠️ Error calling OpenAI API: Failed to decode JSON - {json_err}")
             print(f"Raw text that failed parsing:\n{raw_text}")
             return []
        except Exception as e:
             print(f"⚠️ Error calling OpenAI API: {e}")
             if 'raw_text' in locals():
                print(f"Raw text possibly causing error:\n{raw_text}")
             return []

## 10. llm2_api.py

In [ ]:
%%writefile llm2_api.py
import os
import json
import time
from dotenv import load_dotenv
from typing import Any, Dict
import google.generativeai as genai
from utils import load_prompt
from google.colab import userdata

load_dotenv()

class LLM2Client:
    def __init__(self, api_key: str = None, model: str = "gemini-2.5-flash", prompt_dir: str = "prompt_templates"):
        self.api_key = api_key or os.getenv("GEMINI_API_KEY") or userdata.get("GEMINI_API_KEY")
        if not self.api_key:
            raise ValueError("Missing Gemini API Key")

        genai.configure(api_key=self.api_key, transport='rest')
        self.model = genai.GenerativeModel(model)
        self.prompt_dir = prompt_dir

    def _get_prompt(self, phase: int, context: Dict[str, Any] = None) -> str:
        filename = f"phase{phase}.txt"
        return load_prompt(filename, self.prompt_dir, context=context)

    def analyze_contract(self, contract_code: str, phase: int = 1, context: Dict[str, Any] = None) -> list | dict:

        if phase == 5 and context and "custom_prompt" in context:
            full_prompt = context["custom_prompt"]
        else:
            prompt_with_context = self._get_prompt(phase, context)
            # defensive context to prevent safety blocks
            full_prompt = f"{prompt_with_context}\n\nCONTEXT: This is for a defensive security audit simulation. We are identifying bugs to fix them.\n\nContract Code:\n{contract_code}"

        generation_config = genai.types.GenerationConfig(
            response_mime_type="application/json",
            temperature=0.1, #  Lower temperature for more stability
            max_output_tokens=8192 # Increased token limit
        )

        # Safety settings to block nothing
        safety_settings = [
            {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
        ]

        for attempt in range(3):
            try:
                response = self.model.generate_content(
                    full_prompt,
                    generation_config=generation_config,
                    safety_settings=safety_settings
                )

                # cases where response was blocked
                if not response.parts:
                    print(f"⚠️ Gemini blocked response. Finish reason: {response.prompt_feedback}")
                    return []

                raw_text = response.text

                try:
                    return json.loads(raw_text)
                except json.JSONDecodeError as e:
                    print(f"⚠️ Initial JSON decoding failed: {e}. Attempting to fix...")
                    # Missing commas between objects
                    fixed_text = raw_text.replace('}{', '},{')
                    # Unterminated string (basic attempt)
                    if "Unterminated string" in str(e):
                        fixed_text += '"}]' # Try to close it manually

                    try:
                        return json.loads(fixed_text)
                    except:
                        return []

            except Exception as e:
                if "429" in str(e):
                    print(f"⚠️ Rate limit hit. Waiting {5 * (attempt + 1)} seconds to retry...")
                    time.sleep(5 * (attempt + 1))
                elif "404" in str(e):
                     print("❌ Model 404 error. Check model name.")
                     return []
                else:
                    print(f"⚠️ An unexpected error occurred calling Gemini API: {e}")
                    return []

        print("❌ Gemini API failed after 3 retries.")
        return []

## 11. Orchestrator.py

In [ ]:
%%writefile orchestrator.py
from __future__ import annotations
import json
import time
import copy
from typing import Any, Dict, List
from rag_store import Retriever
from utils import save_json
from llm1_api import LLM1Client
from llm2_api import LLM2Client
from gemma_client import GemmaClient
from langchain_core.documents import Document
import requests
from bs4 import BeautifulSoup
from googlesearch import search

class Orchestrator:
    def __init__(self, retriever: Retriever):
        self.llm1 = LLM1Client()
        self.llm2 = LLM2Client()
        self.gemma = GemmaClient()
        self.rag = retriever

    def run_phased(self, contract_code: str, contract_id: str = None) -> Dict[str, Any]:
        print(f"\n STARTING ANALYSIS FOR: {contract_id}")
        results: Dict[str, Any] = {}
        if contract_id is None:
            contract_id = "default_contract_id"

        # ---------------- Phase 0: Gemma ----------------
        print("🚀 Starting Phase 0: Initial Gemma analysis...")
        try:
            gemma_report = self.gemma.analyze(contract_code)
            results["phase0_gemma"] = {"raw_text": gemma_report}
            gemma_report_text = json.dumps(gemma_report, indent=2)
            if contract_id not in self.rag.registry:
                self.rag.index_contract(contract_id, gemma_report) # Index raw text
        except Exception as e:
            print(f"❌ Error in Phase 0: {e}")
            results["phase0_gemma"] = {}

        rag_context_list = self.rag.retrieve(contract_id=contract_id, query="vulnerabilities")
        rag_text = "\n".join(rag_context_list)

        # ---------------- Phase 1: Initial Analysis ----------------
        print("\n🚀 Starting Phase 1: Initial LLM analysis...")
        try:
            phase1_context = {"gemma_analysis_report": rag_text}
            results["phase1"] = {
                "llm1": self.llm1.analyze_contract(contract_code, phase=1, context=phase1_context),
                "llm2": self.llm2.analyze_contract(contract_code, phase=1, context=phase1_context)
            }
            time.sleep(5)
        except Exception as e:
            print(f"❌ Error in Phase 1: {e}")
            results["phase1"] = {"llm1": [], "llm2": []}

        # ---------------- Phase 2: Cross Verification ----------------
        print("\n🚀 Starting Phase 2: Cross verification...")
        try:
            results["phase2"] = {
                "llm1_on_llm2": self.llm1.analyze_contract(contract_code, phase=2, context={"counterparty_report": results.get("phase1", {}).get("llm2", [])}),
                "llm2_on_llm1": self.llm2.analyze_contract(contract_code, phase=2, context={"counterparty_report": results.get("phase1", {}).get("llm1", [])})
            }
            time.sleep(5)
        except Exception as e:
            print(f"❌ Error in Phase 2: {e}")
            results["phase2"] = {"llm1_on_llm2": [], "llm2_on_llm1": []}

        # ---------------- Phase 3: Consensus ----------------
        print("\n🚀 Starting Phase 3: Consensus...")
        try:
            results["phase3"] = {
                "llm1_on_llm2": self.llm1.analyze_contract(contract_code, phase=3, context={"other_findings": results.get("phase2", {}).get("llm2_on_llm1", [])}),
                "llm2_on_llm1": self.llm2.analyze_contract(contract_code, phase=3, context={"other_findings": results.get("phase2", {}).get("llm1_on_llm2", [])})
            }
            time.sleep(5)
        except Exception as e:
            print(f"❌ Error in Phase 3: {e}")
            results["phase3"] = {"llm1_on_llm2": [], "llm2_on_llm1": []}

        # ---------------- Phase 4: Counter-Example Verification ----------------
        print("\n🚀 Starting Phase 4: Dual-Model Counter-Example Verification...")
        results["phase4"] = []
        try:
            # 1. Aggregate Phase 1 Findings
            phase1_findings = {}
            for finding in results.get("phase1", {}).get("llm1", []) + results.get("phase1", {}).get("llm2", []):
                if isinstance(finding, dict) and "id" in finding:
                    phase1_findings[finding["id"]] = finding

            # 2. Load Local Counter Examples
            try:
                with open("counter_examples.json", "r", encoding="utf-8") as f:
                    counter_examples = json.load(f)
            except:
                counter_examples = {}

            # 3. Collect Candidates
            candidates_for_phase4 = []
            processed_ids = set()
            for key in ["llm1_on_llm2", "llm2_on_llm1"]:
                sublist = results.get("phase3", {}).get(key, [])
                if isinstance(sublist, list):
                    for item in sublist:
                        if isinstance(item, dict) and item.get("stance") == "confirm":
                            if item.get("id") and item.get("id") not in processed_ids:
                                candidates_for_phase4.append(item)
                                processed_ids.add(item.get("id"))

            # Fail-Safe Logic
            if not candidates_for_phase4:
                print("\n🛑 Phase 3 yielded 0 confirmations. ACTIVATING FAIL-SAFE.")
                for vuln_id, finding in phase1_findings.items():
                    severity = finding.get("severity", "").lower()
                    confidence = finding.get("confidence", 0)
                    if severity in ["high", "critical"] or confidence >= 0.8:
                        if vuln_id not in processed_ids:
                            candidates_for_phase4.append({"id": vuln_id, "stance": "confirm"})
                            processed_ids.add(vuln_id)

            print(f"⚡ Verifying {len(candidates_for_phase4)} candidates...")

            for finding in candidates_for_phase4:
                original_finding = phase1_findings.get(finding.get("id"), {})
                vuln_category = original_finding.get("category")
                vuln_name = original_finding.get("title", "Unknown")
                counter = None

                # A. Try Local Lookup
                if vuln_category in counter_examples:
                    counter = counter_examples[vuln_category]
                    print(f"   ✔️ Found local counter-example for {vuln_category}.")

                # B. Try Web Search
                else:
                    print(f"   🚦 No local example for '{vuln_name}'. Attempting web search...")
                    try:
                        search_results = list(search(f'"{vuln_name}" Solidity secure code pattern', num_results=1, lang="en"))
                        if search_results:
                            url = search_results[0]
                            print(f"      🕷️ Crawling: {url}")
                            resp = requests.get(url, timeout=5, headers={'User-Agent': 'Mozilla/5.0'})
                            soup = BeautifulSoup(resp.text, 'html.parser')
                            text = soup.get_text()[:5000]

                            # Use LLM1 (GPT) for Extraction
                            extract = self.llm1.analyze_contract("", phase=5, context={"custom_prompt": f"Extract secure code for {vuln_name} from:\n{text}"})
                            if isinstance(extract, list) and extract: extract = extract[0]
                            if extract.get("code_snippet"):
                                counter = {"code_snippet": extract["code_snippet"], "explanation": "Web extracted"}
                                print("      ✔️ Web extraction successful.")
                    except Exception as e:
                        print(f"      ❌ Web search failed: {e}")

                # C. Fallback to Internal Knowledge Generation
                if not counter:
                    print(f" 🧠 Web search failed. Generating counter-example using GPT (LLM1)...")
                    try:
                        gen_prompt = f"""
                        You are a smart contract security expert.
                        Generate a secure Solidity code snippet and a brief explanation that fixes the vulnerability: "{vuln_name}".
                        Respond ONLY with a valid JSON object: {{ "code_snippet": "...", "explanation": "..." }}
                        """
                        #  LLM1 (GPT) to generate the counter-example
                        gen_result = self.llm1.analyze_contract("", phase=5, context={"custom_prompt": gen_prompt})

                        if isinstance(gen_result, list) and gen_result: gen_result = gen_result[0]

                        if gen_result.get("code_snippet"):
                            counter = {
                                "code_snippet": gen_result["code_snippet"],
                                "explanation": gen_result.get("explanation", "Generated by GPT Internal Knowledge")
                            }
                            print(" ✅ GPT successfully generated a counter-example.")
                    except Exception as e:
                        print(f"  ❌ GPT generation failed: {e}")

                # D. Run Differential Analysis
                if counter:
                    print(f" ⚖️ Running differential analysis with BOTH models...")
                    context_data = {
                        "id": finding.get("id"),
                        "vulnerable_code": original_finding.get("evidence"),
                        "initial_rationale": original_finding.get("rationale"),
                        "patched_code": counter["code_snippet"],
                        "fix_explanation": counter.get("explanation")
                    }

                    verdict1 = self.llm1.analyze_contract("", phase=4, context=context_data)
                    if isinstance(verdict1, list): verdict1 = verdict1[0] if verdict1 else {}

                    verdict2 = self.llm2.analyze_contract("", phase=4, context=context_data)
                    if isinstance(verdict2, list): verdict2 = verdict2[0] if verdict2 else {}

                    results["phase4"].append({
                        "id": finding.get("id"),
                        "llm1_verdict": verdict1,
                        "llm2_verdict": verdict2
                    })
                    time.sleep(2)
                else:
                    print(f" ⚠️ Skipping {vuln_name} (Could not find or generate counter-example).")

            print("✅ Phase 4 completed.")
        except Exception as e:
            print(f"❌ Error in Phase 4: {e}")

        # Build Report
        print("\n🚀 Building final report...")
        try:
            results["final_report"] = self.build_final_report(results)
            save_json(results["final_report"], "outputs/report.json")
            print("✅ Report saved.")
        except Exception as e:
             print(f"❌ Error building report: {e}")
             results["final_report"] = {}

        return results

    def build_final_report(self, results):
        confirmed_vulnerabilities = []
        disputed_vulnerabilities = []
        phase1_map = {}

        # Map Phase 1 findings
        for f in results.get("phase1", {}).get("llm1", []) + results.get("phase1", {}).get("llm2", []):
            if isinstance(f, dict) and "id" in f: phase1_map[f["id"]] = f

        # Process Phase 4
        for entry in results.get("phase4", []):
            vuln_id = entry.get("id")
            v1 = entry.get("llm1_verdict", {})
            v2 = entry.get("llm2_verdict", {})

            dec1 = v1.get("final_decision")
            dec2 = v2.get("final_decision")

            orig = phase1_map.get(vuln_id, {})
            # Prefer LLM1's explanation if available, else LLM2
            primary_verdict = v1 if dec1 == "vulnerability_confirmed" else v2

            raw_cfg = primary_verdict.get("control_flow_graph", "")
            clean_cfg = raw_cfg.replace("```mermaid", "").replace("```", "").strip() if raw_cfg else "graph LR; A[Not provided];"

            vuln_details = {
                "type_of_error": orig.get("title"),
                "code_snippet": orig.get("evidence"),
                "recommendation_suggested_fix": orig.get("recommendation"),
                "vulnerable_flow_steps": primary_verdict.get("vulnerable_flow_steps", []),
                "control_flow_graph": clean_cfg,
                "llm1_decision": dec1,
                "llm2_decision": dec2
            }

            if dec1 == "vulnerability_confirmed" and dec2 == "vulnerability_confirmed":
                vuln_details["status"] = "confirmed"
                vuln_details["confidence"] = 1.0
                vuln_details["evidence_rationale"] = f"**Consensus Reached:** Both models confirmed.\n\n{primary_verdict.get('detailed_reasoning')}"
                confirmed_vulnerabilities.append(vuln_details)
            elif dec1 == "vulnerability_confirmed" or dec2 == "vulnerability_confirmed":
                vuln_details["status"] = "disputed"
                vuln_details["confidence"] = 0.5
                vuln_details["evidence_rationale"] = f"**Disputed:** Models disagreed.\n\nLLM1 (OpenAI): {v1.get('detailed_reasoning', 'Rejected')}\n\nLLM2 (Gemini): {v2.get('detailed_reasoning', 'Rejected')}"
                disputed_vulnerabilities.append(vuln_details)

        summary = {"status": "Analysis Complete", "confirmed_vulnerabilities": confirmed_vulnerabilities, "disputed_vulnerabilities": disputed_vulnerabilities}
        if not confirmed_vulnerabilities and not disputed_vulnerabilities: summary["status"] = "No vulnerabilities confirmed."

        return {
            "summary_report": summary,
            "detailed_llm_analysis": copy.deepcopy({
                "phase0_gemma": results.get("phase0_gemma"),
                "phase1": results.get("phase1"),
                "phase2": results.get("phase2"),
                "phase3": results.get("phase3"),
                "phase4": results.get("phase4")
            })
        }

## 12. frontend.py

In [ ]:
import streamlit as st
import streamlit.components.v1 as components
import json
import os
import time
import requests
import urllib3
import plotly.graph_objects as go

# SSL warning disabling
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


DEFAULT_API_URL = "https://foundations-extensions-aspect-centres.trycloudflare.com/analyze"

# --- PAGE CONFIGURATION ---
st.set_page_config(
    page_title="Smart Contract Security Auditor",
    layout="wide",
    initial_sidebar_state="expanded"
)

# --- CUSTOM CSS (Dark Dashboard Theme) ---
st.markdown("""
<style>
    /* Global Font & Colors */
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600&display=swap');

    html, body, [class*="css"] {
        font-family: 'Inter', sans-serif;
    }

    .stApp {
        background-color: #0E1117; /* Deep Dark Background */
        color: #FAFAFA;
    }

    /* Sidebar Styling */
    section[data-testid="stSidebar"] {
        background-color: #161B22;
        border-right: 1px solid #30333D;
    }

    /* Custom Navigation Buttons (Radio) */
    div[data-testid="stSidebar"] .stRadio div[role="radiogroup"] > label > div:first-child {
        display: none; /* Hide radio circles */
    }
    div[data-testid="stSidebar"] .stRadio label {
        padding: 12px 15px;
        margin: 4px 0;
        border-radius: 6px;
        cursor: pointer;
        color: #8B949E;
        transition: all 0.2s;
        border-left: 3px solid transparent;
    }
    div[data-testid="stSidebar"] .stRadio label:hover {
        background-color: #21262D;
        color: #C9D1D9;
    }
    div[data-testid="stSidebar"] .stRadio label:has(input:checked) {
        background-color: #1F242C;
        color: #58A6FF; /* Active Blue */
        border-left: 3px solid #58A6FF;
        font-weight: 600;
    }

    /* Metric Cards */
    div[data-testid="metric-container"] {
        background-color: #21262D;
        border: 1px solid #30363D;
        padding: 15px;
        border-radius: 8px;
        box-shadow: 0 4px 6px rgba(0,0,0,0.3);
    }
    div[data-testid="metric-container"] label {
        color: #8B949E; /* Muted label color */
    }

    /* Success/Error Alerts */
    .stAlert {
        background-color: #21262D;
        color: #E6EDF3;
        border: 1px solid #30363D;
    }

    /* Center Mermaid Diagrams */
    .mermaid-container {
        display: flex;
        justify-content: center;
        background-color: #0d1117;
        padding: 20px;
        border-radius: 8px;
        border: 1px solid #30363D;
    }
</style>
""", unsafe_allow_html=True)

# --- HELPER 1: Gauge Chart (Security Score) ---
def render_gauge(score):
    """Renders a visually appealing gauge chart using Plotly."""
    # Color Logic: Red (<50), Orange (<80), Green (>=80)
    color = "#DA3633" if score < 50 else "#D29922" if score < 80 else "#2EA043"

    fig = go.Figure(go.Indicator(
        mode = "gauge+number",
        value = score,
        domain = {'x': [0, 1], 'y': [0, 1]},
        title = {'text': "Security Score", 'font': {'size': 20, 'color': "#E6EDF3"}},
        number = {'font': {'color': "#E6EDF3", 'size': 40}},
        gauge = {
            'axis': {'range': [None, 100], 'tickwidth': 1, 'tickcolor': "#30363D"},
            'bar': {'color': color},
            'bgcolor': "#161B22",
            'borderwidth': 0,
            'bordercolor': "#30363D",
            'steps': [
                {'range': [0, 100], 'color': "#161B22"} # Background track
            ],
        }
    ))
    fig.update_layout(
        paper_bgcolor = "#0E1117",
        font = {'color': "#E6EDF3", 'family': "Inter"},
        height=250,
        margin=dict(l=30, r=30, t=50, b=20)
    )
    return fig

# --- HELPER 2: Mermaid Renderer (Iframe) ---
def render_mermaid(code):
    """
    Renders Mermaid code inside an iframe.
    This is MORE STABLE than st_mermaid because it isolates the JS environment.
    """
    html = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <script type="module">
            import mermaid from 'https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs';
            mermaid.initialize({{ startOnLoad: true, theme: 'dark', securityLevel: 'loose' }});
        </script>
        <style>
            body {{ background-color: #0E1117; margin: 0; display: flex; justify-content: center; align-items: center; height: 100vh; }}
            .mermaid {{ text-align: center; width: 100%; }}
        </style>
    </head>
    <body>
        <div class="mermaid">
        {code}
        </div>
    </body>
    </html>
    """
    return html

# --- STATE MANAGEMENT ---
if "api_url" not in st.session_state:
    st.session_state["api_url"] = DEFAULT_API_URL

# --- SIDEBAR ---
with st.sidebar:
    st.markdown("### NAVIGATION")
    page = st.radio("Go to", ["Dashboard", "Audit History", "Settings"], label_visibility="collapsed")

    st.markdown("---")
    st.markdown("### STATUS")

    # Simple Connectivity Check
    if "trycloudflare.com" in st.session_state["api_url"]:
         st.success("🟢 Backend Configured")
         clean_url = st.session_state["api_url"].split("//")[-1].split("/")[0]
         st.caption(f"Host: {clean_url}")
    else:
         st.error("🔴 No API URL")

# --- MAIN CONTENT ---

if page == "Dashboard":
    st.title("🛡️ Smart Contract Security Auditor")
    st.markdown("### Automated Vulnerability Detection Pipeline")

    # 1. File Upload
    uploaded_file = st.file_uploader("Upload Smart Contract (.sol)", type=["sol"])

    contract_code = ""
    if uploaded_file:
        contract_code = uploaded_file.getvalue().decode("utf-8")
        with st.expander("📄 View Source Code"):
            st.code(contract_code, language="solidity")

    # 2. Action Button
    start_btn = st.button("Start Security Audit 🚀", type="primary", use_container_width=True, disabled=not uploaded_file)

    if start_btn and uploaded_file:
        progress_bar = st.progress(0, text="Initializing...")
        status_box = st.empty()

        final_report = None

        try:
            # A. Send Request
            status_box.info("🔄 Sending code to Colab Backend...")
            payload = {"code": contract_code, "id": uploaded_file.name}

            response = requests.post(st.session_state["api_url"], json=payload, timeout=600, verify=False)

            # B. Handle Timeout / Polling
            if response.status_code == 524:
                status_box.warning("⏳ Deep Analysis in Progress (Polling Mode)...")
                report_endpoint = st.session_state["api_url"].replace("/analyze", "/get_report")

                # Poll loop
                for i in range(60):
                    time.sleep(5)
                    prog = 10 + int((i/60)*85)
                    progress_bar.progress(prog, text=f"Phase {min(5, i//12 + 1)}/5: Analyzing...")

                    try:
                        poll = requests.get(report_endpoint, verify=False)
                        if poll.status_code == 200:
                            data = poll.json()
                            if "summary_report" in data:
                                final_report = data
                                progress_bar.progress(100, text="Analysis Complete!")
                                break
                    except: pass
                else:
                    st.error("Timeout: Report could not be retrieved.")
                    st.stop()

            # C. Handle Immediate Success
            elif response.status_code == 200:
                result = response.json()
                # Check for backend errors
                if "error" in result:
                    st.error(f"Backend Error: {result['error']}")
                    st.stop()

                final_report = result.get("final_report", {})
                progress_bar.progress(100, text="Done!")

            else:
                st.error(f"HTTP Error: {response.status_code}")
                st.stop()

            # D. Save & Render
            if final_report:
                # Save locally for History tab
                os.makedirs("outputs", exist_ok=True)
                with open("outputs/report.json", "w") as f:
                    json.dump(final_report, f, indent=2)

                status_box.empty()
                progress_bar.empty()

                # --- DASHBOARD RENDERING ---
                summary = final_report.get("summary_report", {})
                confirmed = summary.get("confirmed_vulnerabilities", [])
                disputed = summary.get("disputed_vulnerabilities", [])
                all_vulns = confirmed + disputed

                # Score Calc
                deduction = (len(confirmed) * 20) + (len(disputed) * 5)
                score = max(0, 100 - deduction)

                # Top Row: Metrics
                c1, c2, c3 = st.columns([1, 1, 1])
                with c1:
                    st.plotly_chart(render_gauge(score), use_container_width=True)
                with c2:
                    st.markdown("### Findings")
                    st.metric("Total Issues", len(all_vulns))
                    st.metric("Confirmed Critical", len(confirmed), delta_color="inverse")
                with c3:
                    st.markdown("### Metadata")
                    st.metric("Lines of Code", len(contract_code.splitlines()))
                    st.download_button("⬇️ Download JSON", data=json.dumps(final_report, indent=2), file_name="report.json", mime="application/json")

                st.divider()

                # Findings List
                st.subheader("Detailed Findings")
                if not all_vulns:
                    st.success("✅ No critical vulnerabilities detected.")
                else:
                    for v in all_vulns:
                        is_confirmed = v.get('status') == 'confirmed'
                        icon = "🔴" if is_confirmed else "🟠"
                        status_text = "CONFIRMED" if is_confirmed else "DISPUTED"

                        with st.expander(f"{icon} {v.get('type_of_error', 'Issue')} — {status_text}", expanded=is_confirmed):
                            t1, t2, t3 = st.tabs(["📝 Details", "👣 Attack Path", "📊 Visual CFG"])

                            with t1:
                                st.markdown("**Description:**")
                                st.info(v.get('evidence_rationale', 'N/A'))
                                st.markdown("**Fix Recommendation:**")
                                st.success(v.get('recommendation_suggested_fix', 'N/A'))
                                st.markdown("**Code Snippet:**")
                                st.code(v.get('code_snippet', '// N/A'), language="solidity")

                            with t2:
                                steps = v.get("vulnerable_flow_steps", [])
                                if steps:
                                    for s in steps:
                                        st.markdown(f"**{s.get('step')}.** {s.get('description')}")
                                        st.code(s.get('code_block', ''), language="solidity")
                                else:
                                    st.caption("No execution path generated.")

                            with t3:
                                cfg = v.get('control_flow_graph', 'graph TD; A[No CFG];')

                                # Render using the iframe helper for robustness
                                components.html(render_mermaid(cfg), height=350, scrolling=True)

        except Exception as e:
            st.error(f"Analysis Failed: {e}")

elif page == "Audit History":
    st.title("📜 Audit History")
    if os.path.exists("outputs/report.json"):
        with open("outputs/report.json", "r") as f:
            st.json(json.load(f))
    else:
        st.info("No reports found.")

elif page == "Settings":
    st.title("⚙️ Settings")
    new_url = st.text_input("Backend API URL", value=st.session_state["api_url"])
    if st.button("Save URL"):
        st.session_state["api_url"] = new_url
        st.success("URL Updated!")

## 13. Connection with ngrok for frontend (skip in case only backend is needed)

In [ ]:
%%writefile api_server.py
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel
import nest_asyncio
import os
import asyncio
import subprocess
import threading
import time
import re

from orchestrator import Orchestrator
from rag_store import Retriever

app = FastAPI(title="Smart Contract Analyzer API")
orchestrator = None

class ContractInput(BaseModel):
    code: str
    id: str = "default_contract_id"

@app.post("/analyze")
async def analyze_contract(input: ContractInput) -> dict:
    global orchestrator
    if orchestrator is None:
        return {"error": "Backend is loading models... please wait."}
    print(f"\nReceived request: {input.id}")
    try:
        report = await asyncio.to_thread(orchestrator.run_phased, input.code, input.id)
        return report
    except Exception as e:
        return {"error": str(e)}

# ENDPOINT: Serve the saved report
@app.get("/get_report")
async def get_report():
    report_path = "outputs/report.json"
    if os.path.exists(report_path):
        return FileResponse(report_path)
    return {"status": "processing", "message": "Report not ready yet."}

# --- Cloudflare Tunnel Helper ---
def start_cloudflared(port):
    if not os.path.exists("cloudflared-linux-amd64"):
        print("Downloading Cloudflare Tunnel...")
        subprocess.run("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", shell=True)
        subprocess.run("chmod +x cloudflared-linux-amd64", shell=True)

    print(f"Starting Cloudflare Tunnel on port {port}...")
    process = subprocess.Popen(
        f"./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:{port}",
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    for line in process.stderr:
        if "trycloudflare.com" in line:
            url = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if url:
                print("\n" + "="*60)
                print(f"🚀 \033[92mPublic URL: {url.group(0)}\033[0m")
                print("="*60 + "\n")
                break

async def run_server():
    global orchestrator
    if orchestrator is None:
        print("Loading Retriever...")
        retriever = Retriever()
        print("Loading Orchestrator...")
        orchestrator = Orchestrator(retriever=retriever)
        print("✅ Backend ready.")

    threading.Thread(target=start_cloudflared, args=(8000,), daemon=True).start()

    nest_asyncio.apply()
    config = uvicorn.Config(app, host="0.0.0.0", port=8000)
    server = uvicorn.Server(config)
    await server.serve()

## Running the app


In [ ]:
!pip install --upgrade huggingface_hub

In [ ]:
import torch
import bitsandbytes as bnb
from transformers import AutoModelForCausalLM

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"BitsAndBytes version: {bnb.__version__}")

try:
    import accelerate
    print(f"Accelerate version: {accelerate.__version__}")
except ImportError:
    print("Accelerate not found!")

print("✅ Libraries loaded successfully. You can proceed to run the rest of the cells.")

In [ ]:
import sys
import gc
import torch

#Force Clean Memory
if 'orchestrator' in globals():
    del orchestrator
if 'retriever' in globals():
    del retriever

gc.collect()
torch.cuda.empty_cache()

print(f"GPU Memory Free: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB")


import asyncio
# Force reload to ensure code updates are applied
if 'api_server' in sys.modules:
    del sys.modules['api_server']

from api_server import run_server

await run_server()